# Transformer

In [8]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader, Subset
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import optuna

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




In [9]:
# Cell 1: 데이터 로딩

# 경로 설정
DATA_DIR = './data_filtering/filtered/'

# train 데이터 로드
train_data = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print("✅ train_data shape:", train_data.shape)
display(train_data.head())

# test 데이터 10개 로드
test_data_list = []
for i in range(10):
    test_path = os.path.join(DATA_DIR, f'TEST_{i:02d}.csv')
    df = pd.read_csv(test_path)
    test_data_list.append(df)
    print(f"✅ Loaded TEST_{i:02d}.csv | shape: {df.shape}")


✅ train_data shape: (102676, 6)


,date_ordinal,date,store_menu,store,menu,sales
0,738521,2023-01-01,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
1,738522,2023-01-02,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
2,738523,2023-01-03,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
3,738524,2023-01-04,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0
4,738525,2023-01-05,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ,1인 수저세트,0


✅ Loaded TEST_00.csv | shape: (5404, 6)
✅ Loaded TEST_01.csv | shape: (5404, 6)
✅ Loaded TEST_02.csv | shape: (5404, 6)
✅ Loaded TEST_03.csv | shape: (5404, 6)
✅ Loaded TEST_04.csv | shape: (5404, 6)
✅ Loaded TEST_05.csv | shape: (5404, 6)
✅ Loaded TEST_06.csv | shape: (5404, 6)
✅ Loaded TEST_07.csv | shape: (5404, 6)
✅ Loaded TEST_08.csv | shape: (5404, 6)
✅ Loaded TEST_09.csv | shape: (5404, 6)


In [10]:
# Cell 2: 전처리 함수 정의 및 실행
def preprocess_data(df):
    # 날짜형 변환
    df['date'] = pd.to_datetime(df['date'])
    df['date_ordinal'] = df['date'].map(pd.Timestamp.toordinal)

    # 'store_menu' 식별자 추가 (이미 있는 경우 생략 가능)
    if 'store_menu' not in df.columns:
        df['store_menu'] = df['store'] + "_" + df['menu']

    # 날짜 관련 파생 변수
    df['day_of_week'] = df['date'].dt.dayofweek      # 0=월 ~ 6=일
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day

    # 필요한 feature만 추출
    use_cols = [
        'date', 'date_ordinal', 'store_menu', 'sales',
        'day_of_week', 'is_weekend', 'month', 'day'
    ]
    return df[use_cols]

# train 데이터 전처리
train_data = preprocess_data(train_data)
display(train_data.head())

# test 데이터 전처리
for i in range(10):
    test_data_list[i] = preprocess_data(test_data_list[i])


,date,date_ordinal,store_menu,sales,day_of_week,is_weekend,month,day
0,2023-01-01,738521,느티나무 셀프BBQ_1인 수저세트,0,6,1,1,1
1,2023-01-02,738522,느티나무 셀프BBQ_1인 수저세트,0,0,0,1,2
2,2023-01-03,738523,느티나무 셀프BBQ_1인 수저세트,0,1,0,1,3
3,2023-01-04,738524,느티나무 셀프BBQ_1인 수저세트,0,2,0,1,4
4,2023-01-05,738525,느티나무 셀프BBQ_1인 수저세트,0,3,0,1,5


In [11]:
# Cell 3: Transformer 모델 정의
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim, d_model, nhead, num_layers, dropout=0.1, output_len=7):
        super(TimeSeriesTransformer, self).__init__()
        
        self.model_type = 'Transformer'
        self.output_len = output_len
        self.d_model = d_model

        # 입력 임베딩: Linear -> Positional Encoding
        self.input_projection = nn.Linear(input_dim, d_model)

        # 포지셔널 인코딩
        self.positional_encoding = self._generate_positional_encoding(1000, d_model)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 디코더 (예측용)
        self.decoder = nn.Linear(d_model, output_len)

    def forward(self, src):
        # src: [B, L, input_dim]
        src = self.input_projection(src)  # [B, L, d_model]

        pos_encoding = self.positional_encoding[:src.size(1), :].unsqueeze(0).to(src.device)  # [1, L, d_model]
        src = src + pos_encoding  # [B, L, d_model]

        src = src.permute(1, 0, 2)  # [L, B, d_model]
        output = self.transformer_encoder(src)  # [L, B, d_model]
        output = output.permute(1, 0, 2)  # [B, L, d_model]

        last_hidden = output[:, -1, :]  # [B, d_model]
        pred = self.decoder(last_hidden)  # [B, 7]
        return pred


    def _generate_positional_encoding(self, max_len, d_model):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe  # shape: [max_len, d_model] ← [1, max_len, d_model] ❌ X


In [12]:
# ✅ SequenceDataset 정의 (유지)
class SequenceDataset(Dataset):
    def __init__(self, df, input_len=28, output_len=7):
        self.input_len = input_len
        self.output_len = output_len
        self.data = []

        df = df.sort_values(['store_menu', 'date'])  # 시계열 정렬

        for sm in df['store_menu'].unique():
            df_sm = df[df['store_menu'] == sm]
            values = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values
            for i in range(max(0, len(values) - input_len - output_len + 1)):
                x = values[i:i+input_len]
                y = values[i+input_len:i+input_len+output_len, 0]
                self.data.append((x, y))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x, y = self.data[idx]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

# ✅ 1. 전체 시퀀스 생성
full_dataset = SequenceDataset(train_data)

# ✅ 2. 전체 시퀀스에 대해 KFold split
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(np.arange(len(full_dataset))))  # 시퀀스 인덱스를 기반으로 분할

# ✅ 3. Fold별로 DataLoader 생성
for fold, (train_idx, val_idx) in enumerate(folds):
    print(f"\n🌀 Fold {fold+1}")

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

    print(f"✅ Fold {fold+1} | Train samples: {len(train_subset)} | Val samples: {len(val_subset)}")

    # 🔁 모델 학습/검증 루프 연결 가능



🌀 Fold 1
✅ Fold 1 | Train samples: 76891 | Val samples: 19223

🌀 Fold 2
✅ Fold 2 | Train samples: 76891 | Val samples: 19223

🌀 Fold 3
✅ Fold 3 | Train samples: 76891 | Val samples: 19223

🌀 Fold 4
✅ Fold 4 | Train samples: 76891 | Val samples: 19223

🌀 Fold 5
✅ Fold 5 | Train samples: 76892 | Val samples: 19222


## OPTUNA 써서 튜닝 (더 수정해봐야됨)
- kfold 파라미터 설정
- optuna로 파라미터 최적화 후, k-fold validation 재수행해서 모델 학습하는 것도 고려

In [ ]:
NUM_EPOCHS = 10
SAVE_DIR = './optuna_models'
os.makedirs(SAVE_DIR, exist_ok=True)

def objective(trial):
    d_model = trial.suggest_categorical('d_model', [32, 64, 128])
    nhead = trial.suggest_categorical('nhead', [2, 4, 8])
    num_layers = trial.suggest_int('num_layers', 1, 3)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)

    fold_val_losses = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=64, shuffle=True)
        val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=64, shuffle=False)

        model = TimeSeriesTransformer(5, d_model, nhead, num_layers, output_len=7).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        for epoch in range(NUM_EPOCHS):
            model.train()
            for x_batch, y_batch in train_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                optimizer.zero_grad()
                output = model(x_batch)
                loss = criterion(output, y_batch)

                # NaN 방지
                if not torch.isfinite(loss):
                    print("❌ NaN loss 발생, trial 중단")
                    return float("inf")

                loss.backward()
                optimizer.step()

        # validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                output = model(x_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * x_batch.size(0)

        val_loss /= len(val_loader.dataset)
        fold_val_losses.append(val_loss)

    avg_val_loss = sum(fold_val_losses) / len(fold_val_losses)

    # 모델 저장 (best_trial은 나중에 밖에서 판단)
    save_path = os.path.join(SAVE_DIR, f'model_trial_{trial.number}.pt')
    torch.save(model.state_dict(), save_path)
    trial.set_user_attr("model_path", save_path)

    return avg_val_loss



In [ ]:
# ✅ 튜닝 실행 및 결과 확인
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ✅ 최적 결과 출력
print("🎯 Best hyperparameters:")
print(study.best_params)
print(f"📉 Best validation loss: {study.best_value:.4f}")
print(f"💾 Best model saved to: {study.best_trial.user_attrs['best_model_path']}")

# 현재까지 trial 7까지 103m


[I 2025-08-07 17:03:56,281] A new study created in memory with name: no-name-453ce64c-3b3a-4c63-bf86-609dfc0e71cd
[I 2025-08-07 17:19:55,154] Trial 0 finished with value: 902.3558286266473 and parameters: {'d_model': 32, 'nhead': 8, 'num_layers': 3, 'lr': 0.00022185522296289448}. Best is trial 0 with value: 902.3558286266473.
[I 2025-08-07 17:26:54,166] Trial 1 finished with value: inf and parameters: {'d_model': 128, 'nhead': 4, 'num_layers': 3, 'lr': 0.003041200302436175}. Best is trial 0 with value: 902.3558286266473.


❌ NaN loss 발생, trial 중단


[I 2025-08-07 17:43:02,303] Trial 2 finished with value: 853.0949108999396 and parameters: {'d_model': 64, 'nhead': 2, 'num_layers': 3, 'lr': 0.00018371343592170412}. Best is trial 2 with value: 853.0949108999396.
[I 2025-08-07 17:55:53,083] Trial 3 finished with value: 923.1002432612004 and parameters: {'d_model': 32, 'nhead': 2, 'num_layers': 2, 'lr': 0.00021793094884057157}. Best is trial 2 with value: 853.0949108999396.
[I 2025-08-07 18:08:43,239] Trial 4 finished with value: 828.3532950460665 and parameters: {'d_model': 32, 'nhead': 2, 'num_layers': 2, 'lr': 0.0003368619272701501}. Best is trial 4 with value: 828.3532950460665.
[I 2025-08-07 18:24:39,186] Trial 5 finished with value: 815.303005368044 and parameters: {'d_model': 32, 'nhead': 2, 'num_layers': 3, 'lr': 0.0007299356483708963}. Best is trial 5 with value: 815.303005368044.
[I 2025-08-07 18:37:29,997] Trial 6 finished with value: 908.5753551426876 and parameters: {'d_model': 32, 'nhead': 4, 'num_layers': 2, 'lr': 0.0035

❌ NaN loss 발생, trial 중단


In [ ]:
# 시각화
import optuna.visualization as vis

# ✅ 최적화 history
fig1 = vis.plot_optimization_history(study)
fig1.show()

# ✅ 파라미터 중요도
fig2 = vis.plot_param_importances(study)
fig2.show()


In [ ]:
# best 모델이 다음 경로에 저장되었을 때
# best_path = study.best_trial.user_attrs["best_model_path"]

# ✅ best 모델 로드
best_params = study.best_params
best_path = study.best_trial.user_attrs["best_model_path"]

best_model = TimeSeriesTransformer(
    input_dim=5,
    d_model=best_params['d_model'],
    nhead=best_params['nhead'],
    num_layers=best_params['num_layers'],
    output_len=7
).to(device)

best_model.load_state_dict(torch.load(best_path, map_location=device))
best_model.eval()
print(f"✅ Best model loaded from: {best_path}")


In [ ]:
def predict_test(test_df, model, device='cpu', input_len=28):
    preds = {}

    store_menus = test_df['store_menu'].unique()
    for sm in store_menus:
        df_sm = test_df[test_df['store_menu'] == sm].sort_values('date')
        x = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values[-input_len:]

        if x.shape[0] < input_len:
            continue  # skip 부족한 시계열

        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 28, 5]
        with torch.no_grad():
            output = model(x_tensor)  # [1, 7]
        preds[sm] = output.squeeze(0).cpu().numpy()
    
    return preds  # Dict[str, np.ndarray]


## OPTUNA 적용 안했을 때

In [14]:
# Cell 5: K-Fold 기반 모델 학습 루프
import torch.optim as optim
from tqdm import tqdm

# 하이퍼파라미터
input_dim = 5
d_model = 64
nhead = 4
num_layers = 2
output_len = 7
epochs = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# K-Fold 학습 루프
for fold, (train_idx, val_idx) in enumerate(folds):
    print(f"\n🌀 Fold {fold+1}")

    # ✅ 데이터로더 정의
    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)
    train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=64, shuffle=False)

    # ✅ 모델 초기화
    model = TimeSeriesTransformer(input_dim, d_model, nhead, num_layers, output_len=output_len).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # ✅ Epoch 학습
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0
        for x_batch, y_batch in tqdm(train_loader, desc=f"[Fold {fold+1}][Epoch {epoch}] Training", leave=False):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            output = model(x_batch)  # shape: [B, 7]
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x_batch.size(0)

        train_loss /= len(train_loader.dataset)

        # ✅ 검증
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                output = model(x_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * x_batch.size(0)

        val_loss /= len(val_loader.dataset)

        print(f"✅ [Fold {fold+1}] Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")


/home/wonjun/.local/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")



🌀 Fold 1


✅ [Fold 1] Epoch 01 | Train Loss: 1332.6691 | Val Loss: 1008.4100


✅ [Fold 1] Epoch 02 | Train Loss: 1030.1410 | Val Loss: 888.0281


✅ [Fold 1] Epoch 03 | Train Loss: 912.0716 | Val Loss: 817.4773


...

✅ [Fold 5] Epoch 26 | Train Loss: 751.1077 | Val Loss: 920.7763


✅ [Fold 5] Epoch 27 | Train Loss: 749.4830 | Val Loss: 801.1942


✅ [Fold 5] Epoch 28 | Train Loss: 741.4359 | Val Loss: 784.2410


✅ [Fold 5] Epoch 29 | Train Loss: 731.7579 | Val Loss: 843.3383


✅ [Fold 5] Epoch 30 | Train Loss: 749.2704 | Val Loss: 831.7951


In [15]:
# Cell 6: 테스트 데이터 추론 함수
def predict_test_data(model, test_df, input_len=28, device='cpu'):
    model.eval()
    preds = {}
    
    store_menus = test_df['store_menu'].unique()

    for sm in store_menus:
        df_sm = test_df[test_df['store_menu'] == sm].sort_values('date')
        x = df_sm[['sales', 'day_of_week', 'is_weekend', 'month', 'day']].values[-input_len:]
        
        if x.shape[0] != input_len:
            # 입력 길이가 부족할 경우 패스
            continue

        x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0).to(device)  # [1, 28, 5]
        with torch.no_grad():
            pred = model(x_tensor)  # [1, 7]
        preds[sm] = pred.squeeze(0).cpu().numpy()
    
    return preds  # Dict[str, np.ndarray]

# 모든 test 파일에 대해 예측 수행
all_test_preds = {}

for i in range(10):
    test_df = test_data_list[i]
    pred_dict = predict_test_data(model, test_df, device=device)
    all_test_preds[f'TEST_{i:02d}'] = pred_dict
    print(f"✅ Predicted TEST_{i:02d}: {len(pred_dict)} store_menus")


✅ Predicted TEST_00: 193 store_menus
✅ Predicted TEST_01: 193 store_menus
✅ Predicted TEST_02: 193 store_menus
✅ Predicted TEST_03: 193 store_menus
✅ Predicted TEST_04: 193 store_menus
✅ Predicted TEST_05: 193 store_menus
✅ Predicted TEST_06: 193 store_menus
✅ Predicted TEST_07: 193 store_menus
✅ Predicted TEST_08: 193 store_menus
✅ Predicted TEST_09: 193 store_menus


In [16]:
all_test_preds

{'TEST_00': {'느티나무 셀프BBQ_1인 수저세트': array([3.7223463, 4.047788 , 3.9320502, 4.5087395, 4.2048354, 3.9810188,
         3.7819288], dtype=float32),
  '느티나무 셀프BBQ_BBQ55(단체)': array([26.878405, 25.978308, 25.052307, 25.20486 , 24.81913 , 24.732162,
         24.804518], dtype=float32),
  '느티나무 셀프BBQ_대여료 30,000원': array([-3.8808289, -3.18732  , -3.0444016, -2.3272474, -2.5787516,
         -2.8236969, -3.118274 ], dtype=float32),
  '느티나무 셀프BBQ_대여료 60,000원': array([-5.2751884, -4.5165405, -4.326692 , -3.5835445, -3.8235912,
         -4.0708575, -4.3835897], dtype=float32),
  '느티나무 셀프BBQ_대여료 90,000원': array([-6.4215717, -5.6126285, -5.384797 , -4.620176 , -4.8482313,
         -5.095297 , -5.423605 ], dtype=float32),
  '느티나무 셀프BBQ_본삼겹 (단품,실내)': array([-6.196621 , -5.3972335, -5.1767864, -4.416356 , -4.647041 ,
         -4.8943725, -5.219577 ], dtype=float32),
  '느티나무 셀프BBQ_스프라이트 (단체)': array([16.55646 , 16.221687, 15.660211, 16.001518, 15.639639, 15.479162,
         15.433284], dtype=float32),
  

In [17]:
# Cell 7: 음수 예측값 → 0으로 클리핑
for test_key in all_test_preds:
    for sm_key in all_test_preds[test_key]:
        all_test_preds[test_key][sm_key] = np.clip(all_test_preds[test_key][sm_key], a_min=0, a_max=None)

print("✅ 모든 음수 예측값을 0으로 변환 완료")


✅ 모든 음수 예측값을 0으로 변환 완료


In [18]:
# Cell 8: 예측 결과를 sample_submission에 채워넣기
import pandas as pd

# sample submission 불러오기
submission = pd.read_csv('./result/sample_submission.csv', index_col=0)
print("✅ 불러온 submission shape:", submission.shape)

# 예측값 채워넣기
for test_key in all_test_preds:  # e.g., TEST_00
    for store_menu in all_test_preds[test_key]:
        preds = all_test_preds[test_key][store_menu]  # shape: (7,)
        for day_offset in range(7):
            row_idx = f"{test_key}+{day_offset+1}일"
            if store_menu in submission.columns:
                submission.at[row_idx, store_menu] = preds[day_offset]

# 최종 확인
display(submission.head(10))


✅ 불러온 submission shape: (70, 193)


/tmp/ipykernel_225707/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '3.722346305847168' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_225707/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '26.87840461730957' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/ipykernel_225707/579479100.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '16.556459426879883' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  submission.at[row_idx, store_menu] = preds[day_offset]
/tmp/i

,느티나무 셀프BBQ_1인 수저세트,느티나무 셀프BBQ_BBQ55(단체),"느티나무 셀프BBQ_대여료 30,000원","느티나무 셀프BBQ_대여료 60,000원","느티나무 셀프BBQ_대여료 90,000원","느티나무 셀프BBQ_본삼겹 (단품,실내)",느티나무 셀프BBQ_스프라이트 (단체),느티나무 셀프BBQ_신라면,느티나무 셀프BBQ_쌈야채세트,느티나무 셀프BBQ_쌈장,...,화담숲주막_스프라이트,화담숲주막_참살이 막걸리,화담숲주막_찹쌀식혜,화담숲주막_콜라,화담숲주막_해물파전,화담숲카페_메밀미숫가루,화담숲카페_아메리카노 HOT,화담숲카페_아메리카노 ICE,화담숲카페_카페라떼 ICE,화담숲카페_현미뻥스크림
영업일자,,,,,,,,,,,,,,,,,,,,,
TEST_00+1일,3.722346,26.878405,0.000000,0,0,0,16.556459,0,0.0,0,...,0.0,9.957039,8.135344,0.0,57.560596,43.433723,0.0,42.836849,0.000000,21.498844
TEST_00+2일,4.047788,25.978308,0.000000,0,0,0,16.221687,0,0.0,0,...,0.0,9.967438,8.238888,0.0,54.832172,41.557442,0.0,41.001629,0.305461,20.887962
TEST_00+3일,3.932050,25.052307,0.000000,0,0,0,15.660211,0,0.0,0,...,0.0,9.636678,7.971238,0.0,52.802784,40.037708,0.0,39.503960,0.323982,20.151600
TEST_00+4일,4.508739,25.204861,0.000000,0,0,0,16.001518,0,0.0,0,...,0.0,10.098717,8.466770,0.0,52.390343,39.886253,0.0,39.363499,0.973307,20.402691
TEST_00+5일,4.204835,24.819130,0.000000,0,0,0,15.639639,0,0.0,0,...,0.0,9.761655,8.138510,0.0,52.030018,39.505936,0.0,38.979633,0.695076,20.031395
TEST_00+6일,3.981019,24.732162,0.000000,0,0,0,15.479162,0,0.0,0,...,0.0,9.564416,7.932598,0.0,52.251293,39.579063,0.0,39.043560,0.459071,19.909126
TEST_00+7일,3.781929,24.804518,0.000000,0,0,0,15.433284,0,0.0,0,...,0.0,9.441405,7.787588,0.0,52.660976,39.835545,0.0,39.293472,0.211067,19.919813
TEST_01+1일,0.346145,1.870413,0.722677,0,0,0,0.000000,0,0.0,0,...,0.0,0.285539,0.000000,0.0,31.795727,31.121351,0.0,31.887424,0.000000,19.342705
TEST_01+2일,0.835086,2.285775,1.192987,0,0,0,0.000000,0,0.0,0,...,0.0,0.776810,0.118952,0.0,30.597616,29.960997,0.0,30.682343,0.000000,18.847906


In [19]:
# 저장
submission.to_csv('./result/Vanilla_transformer_kfold_epoch30.csv')
print("✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer_kfold_epoch30.csv")


✅ 최종 제출 파일 저장 완료: ./result/Vanilla_transformer_kfold_epoch30.csv
